#  Week 10, Day 1 Lab — Build Your First AI Agent
### From a stateless chatbot to a working ReAct loop, in 3 hours

**This is a build lab, not a run-and-watch lab.** After Section 2's one worked example, you will
write every tool, every schema, and the agent loop itself from a spec — not fill in one blank in
code that's already written for you.

**How to use this notebook:**
- Cells marked **✍️ WRITE THIS** have only a function signature / docstring / comments — you write
  the body.
- Cells marked **🔧 TODO** ask you to modify or extend something that's partially there.
- Plain cells (no marker) are given to you — API setup, test calls, etc. — run them as-is.


---
## 🛠️ Section 0 — Setup

We'll use **Gemini 2.5 Flash** via Google's `google-genai` SDK — free, no credit card required.

**Get your free API key (2 minutes):**
1. Go to **https://aistudio.google.com/apikey**
2. Sign in with any Google account
3. Click **Create API key** → copy it

> **Free tier note:** there's a rate limit (a handful of requests/minute). If a cell errors with
> a rate-limit message, wait ~30-60 seconds and re-run it.


In [1]:
# Install Google's official GenAI SDK
!pip install -q google-genai


In [8]:
import os
import json
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")


Enter your Gemini API key: ··········


In [9]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.5-flash-lite"
# Sanity check — if this prints a reply, you're good to go
test = client.models.generate_content(
    model=MODEL,
    contents="Say 'Agent lab is ready!' and nothing else."
)
print(test.text)


Agent lab is ready!


---
##  Section 1 — Why Do We Need Agents? (15 minutes)

Run the cell below and see what happens when you ask a plain LLM call something it can't know.


In [10]:
response = client.models.generate_content(
    model=MODEL,
    contents="What's the exact current temperature in Lahore right now?"
)
print(response.text)


I do not have access to real-time weather information to provide the current temperature in Lahore. Please check a weather website or app for the most up-to-date conditions.


Did the model admit it doesn't know, guess, or
make something up? What would it need to answer correctly?




In [ ]:
#answer here

Now let's prove the model is also **stateless**.

In [11]:

r1 = client.models.generate_content(model=MODEL, contents="My name is Fatima. Remember that.")
print("Response 1:", r1.text)

r2 = client.models.generate_content(model=MODEL, contents="What's my name?")
print("Response 2:", r2.text)


Response 1: Nice to meet you, Fatima! I've made a note of it. How can I help you today?
Response 2: I don't know your name yet! You haven't told me. What should I call you?





The model had no idea, because Call 2 didn't include Call 1's messages. Build a `contents` list
below with **both** turns, then ask "What's my name?" as a 3rd turn. Gemini expects alternating
`user` / `model` turns, each shaped like `{"role": ..., "parts": [{"text": ...}]}`.


In [12]:
# build `contents` as a list of 3 turns (user, model, user)
# so the model can correctly answer "What's my name?"

contents = [
    {"role": "user", "parts": [{"text": "Hi, my name is Asma"}]},
    {"role": "model", "parts": [{"text": "Hello Asma! Nice to meet you."}]},
    {"role": "user", "parts": [{"text": "What's my name?"}]}
]

r3 = client.models.generate_content(model=MODEL, contents=contents)
print(r3.text)


Your name is Asma!


**Key takeaway:** every "memory" an LLM app has is really just *you* re-sending history on
every call. This is the backbone of everything below.


---
##  Section 2 — Function Calling Basics: One Worked Example (30 minutes)

We'll build **one** tool together, fully worked, so you see the whole pattern end to end.
Everything after this section, you build yourself from a spec.

**The pattern has 4 parts:**
1. A real Python function that does the work
2. A schema (`FunctionDeclaration`) describing it to the model
3. Send a message with the tool attached — the model asks to call it, it doesn't call it itself
4. Your code executes the function and sends the result back as an "Observation"


In [13]:
# GIVEN — Part 1: the real function
def get_weather(city: str, date: str = "today") -> dict:
    """Pretend weather API — hardcoded data so we don't need a real key."""
    fake_data = {
        "lahore":  {"forecast": "sunny", "temp_c": 34},
        "boston":  {"forecast": "rainy", "temp_c": 18},
        "karachi": {"forecast": "humid", "temp_c": 31},
        "london":  {"forecast": "cloudy", "temp_c": 16},
    }
    data = fake_data.get(city.lower(), {"forecast": "unknown", "temp_c": None})
    return {"city": city, "date": date, **data}

get_weather("Lahore")


{'city': 'Lahore', 'date': 'today', 'forecast': 'sunny', 'temp_c': 34}

In [14]:
# GIVEN — Part 2: the schema
get_weather_declaration = types.FunctionDeclaration(
    name="get_weather",
    description="Get the current or forecasted weather for a specific city.",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. Lahore"},
            "date": {"type": "string", "description": "YYYY-MM-DD, or 'today'/'tomorrow'"}
        },
        "required": ["city"]
    }
)

weather_tool = types.Tool(function_declarations=[get_weather_declaration])
config = types.GenerateContentConfig(tools=[weather_tool])


In [15]:
# GIVEN — Part 3: send a message, the model asks to call the tool instead of answering
chat = client.chats.create(model=MODEL, config=config)
response = chat.send_message("What should I wear in Lahore today?")

part = response.candidates[0].content.parts[0]
print("Did the model ask for a tool?", part.function_call is not None)
print(part.function_call)


Did the model ask for a tool? True
id='call_346692' args={'city': 'Lahore', 'date': 'today'} name='get_weather' partial_args=None will_continue=None


### Part 4: execute the tool and send the result back

`part.function_call` has a `.name` and `.args`. You need to:
1. Pull out the function name and arguments
2. Call the **real** `get_weather` function with those arguments
3. Wrap the result in `types.Part.from_function_response(name=..., response={"result": ...})`
4. Send that part back with `chat.send_message(...)`
5. Print the final text answer

Write it below — no scaffolding this time except the comments telling you the steps.


In [16]:
# complete steps 1-5 described above
function_call = part.function_call

# 1. get the name and args
name = function_call.name
args = function_call.args

# 2. call the real function
result = get_weather(**args)

# 3. wrap the result
function_response_part = types.Part.from_function_response(name=name, response=result)
# 4. send it back
response2 = chat.send_message(function_response_part)

# 5. print the final answer
print(response2.text)

The weather in Lahore today is sunny with a temperature of 34°C. 

Since it's quite warm, you should wear lightweight, breathable clothing made of natural fabrics like cotton or linen. Light colors will also help keep you cool. Don't forget sunglasses, sunscreen, and a hat if you'll be spending time outdoors!


**Test it:** if you did it right, the final answer should mention Lahore's actual sunny/34°C
weather, not a generic answer. If it errors, check the Appendix for the worked solution before
moving on — don't spend more than 5 extra minutes stuck here.


---
##  Section 3 — Build the ReAct Loop Yourself

Section 2 only handles **one** tool call. A real agent loops: call a tool, look at the result,
decide if it needs another tool, and only stop when it has a final answer.

**You are writing this loop from scratch.** Here's the spec:

```
function run_agent(user_query, config, tool_registry, max_steps=5, verbose=True):
    start a chat session with the given config
    send the user_query as the first message

    repeat up to max_steps times:
        look at the parts of the latest response
        collect every part that has a function_call  (there can be more than one!)

        if there are no function_calls:
            print "[Step N] Thought: I have enough info."   (if verbose)
            return the response's text — this is the final answer

        otherwise, for EACH function_call:
            print "[Step N] Action: name(args)"              (if verbose)
            look up the function in tool_registry and call it with the args
            print "[Step N] Observation: result"             (if verbose)
            wrap the result as a function_response Part

        send ALL the function_response Parts back in one chat.send_message() call
        (this becomes the new "response" for the next loop iteration)

    if we run out of steps, return a message saying so
```

Note: Gemini can ask for **multiple tools in one turn** — that's why you collect a *list* of
function_calls per step, not just one.


In [17]:
def run_agent(user_query, config, tool_registry, max_steps=5, verbose=True):
    chat = client.chats.create(model=MODEL, config=config)
    response = chat.send_message(user_query)

    for step in range(max_steps):
        parts = response.candidates[0].content.parts
        function_calls = [p.function_call for p in parts if p.function_call]

        if not function_calls:
            if verbose:
                print(f"[Step {step+1}] Thought: I have enough info.")
            return response.text

        function_response_parts = []
        for fc in function_calls:
            name = fc.name
            args = dict(fc.args)
            if verbose:
                print(f"[Step {step+1}] Action: {name}({args})")
            result = tool_registry[name](**args)
            if verbose:
                print(f"[Step {step+1}] Observation: {result}")
            function_response_parts.append(
                types.Part.from_function_response(name=name, response={"result": result})
            )

        response = chat.send_message(function_response_parts)

    return "Reached max steps without a final answer."

In [29]:
tool_registry = {"get_weather": get_weather}

**Test your loop:**


In [34]:
answer = run_agent("What's the weather like in Karachi and should I bring an umbrella?",
                    config=config, tool_registry=tool_registry)
print()
print("=" * 50)
print("FINAL ANSWER:", answer)


[Step 1] Action: get_weather({'city': 'Karachi', 'date': 'today'})
[Step 1] Observation: {'city': 'Karachi', 'date': 'today', 'forecast': 'humid', 'temp_c': 31}
[Step 2] Action: suggest_activity({'weather_condition': 'humid'})
[Step 2] Observation: {'weather_condition': 'humid', 'suggestion': 'Stay hydrated and relax indoors with AC.'}
[Step 3] Thought: I have enough info.

FINAL ANSWER: The weather in Karachi today is warm and humid with a temperature of 31°C. Since the forecast is humid rather than rainy, you likely won't need an umbrella!


**🔧 Checkpoint (3 min):** test that your loop correctly handles a query needing **no** tool
(should print zero `[Step N] Action:` lines and go straight to a final answer).


In [31]:
answer = run_agent("What's the capital of Pakistan?", config=config, tool_registry=tool_registry)
print(answer)


[Step 1] Thought: I have enough info.
The capital of Pakistan is Islamabad.


---
##  Section 4 — Write Two New Tools From a Spec

Your loop from Section 3 works with *any* tools you register with it — that's the whole point of
building it generically. Now prove it: write two brand-new tools, start to finish, no scaffolding.

### Tool A: `calculate`

**Spec:**
- Signature: `calculate(expression: str) -> dict`
- Takes a basic arithmetic expression as a string, e.g. `"12 * 3 + 7"`
- Only allow digits, `+ - * / ( ) .` and spaces — reject anything else (hint: `re.fullmatch` with
  a character-class pattern) and return `{"error": "..."}` if rejected
- Otherwise, evaluate it (hint: Python's `eval()` works fine for this trusted lab context) and
  return `{"expression": expression, "result": <the number>}`
- Wrap the eval in `try/except` and return `{"error": str(e)}` if it fails


In [21]:
# calculate(expression) -> dict, per the spec above
import re

def calculate(expression: str) -> dict:
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return {"error": "Invalid characters in expression"}
    try:
        result = eval(expression)
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"error": str(e)}


print(calculate("12 * 3 + 7"))
print(calculate("import os"))

{'expression': '12 * 3 + 7', 'result': 43}
{'error': 'Invalid characters in expression'}


### Tool B: `suggest_activity`

**Spec:**
- Signature: `suggest_activity(weather_condition: str) -> dict`
- Takes a weather keyword (`"sunny"`, `"rainy"`, `"humid"`, `"cloudy"`) and returns a dict with
  keys `"weather_condition"` and `"suggestion"` (a one-line activity idea you write yourself)
- If the keyword isn't one you handled, return a sensible fallback suggestion instead of erroring


In [22]:
# suggest_activity(weather_condition) -> dict, per the spec above

def suggest_activity(weather_condition: str) -> dict:
    suggestions = {
        "sunny": "Great day for a walk in the park.",
        "rainy": "Good time to read a book indoors.",
        "humid": "Stay hydrated and relax indoors with AC.",
        "cloudy": "Nice weather for a casual outdoor stroll."
    }
    suggestion = suggestions.get(
        weather_condition.lower(),
        "Check local conditions and plan accordingly."
    )
    return {"weather_condition": weather_condition, "suggestion": suggestion}


print(suggest_activity("rainy"))
print(suggest_activity("blizzard"))   # unhandled keyword — should NOT crash


{'weather_condition': 'rainy', 'suggestion': 'Good time to read a book indoors.'}
{'weather_condition': 'blizzard', 'suggestion': 'Check local conditions and plan accordingly.'}


### Now write the schemas for both

**Spec for each `FunctionDeclaration`:** `name` matching the function, a specific one-sentence
`description` (this is what the model uses to choose the right tool — vague descriptions cause
wrong tool choices), and a `parameters` object schema matching the function's arguments.


In [23]:
# FunctionDeclaration for calculate

calculate_declaration = types.FunctionDeclaration(
    name="calculate",
    description="Evaluate a basic arithmetic expression and return the numeric result.",
    parameters={
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "Arithmetic expression, e.g. '12 * 3 + 7'"}
        },
        "required": ["expression"]
    }
)



In [24]:
# FunctionDeclaration for suggest_activity
suggest_activity_declaration = types.FunctionDeclaration(
    name="suggest_activity",
    description="Suggest a one-line activity idea based on a given weather condition.",
    parameters={
        "type": "object",
        "properties": {
            "weather_condition": {"type": "string", "description": "Weather keyword, e.g. 'sunny', 'rainy'"}
        },
        "required": ["weather_condition"]
    }
)


### Wire everything together and test chaining

Register both new tools (plus `get_weather` from before) into one `Tool`/config, and one
`tool_registry` dict — same pattern as Section 2, now with 3 entries instead of 1.


In [33]:
# build multi_tool (a types.Tool with all 3 declarations),
# a GenerateContentConfig from it, and a tool_registry dict with all 3 functions
multi_tool = types.Tool(function_declarations=[
    get_weather_declaration, calculate_declaration, suggest_activity_declaration
])
config = types.GenerateContentConfig(tools=[multi_tool])
tool_registry = {
    "get_weather": get_weather,
    "calculate": calculate,
    "suggest_activity": suggest_activity
}


In [26]:
# Once wired correctly, this should call get_weather THEN suggest_activity, in that order,
# without you telling it the order — the model figures out the chain itself.

answer = run_agent(
    "What's the weather in Lahore right now, and suggest something fun to do given that weather?",
    config=config, tool_registry=tool_registry
)
print()
print("=" * 50)
print("FINAL ANSWER:", answer)


[Step 1] Action: get_weather({'date': 'today', 'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'date': 'today', 'forecast': 'sunny', 'temp_c': 34}
[Step 2] Action: suggest_activity({'weather_condition': 'sunny'})
[Step 2] Observation: {'weather_condition': 'sunny', 'suggestion': 'Great day for a walk in the park.'}
[Step 3] Thought: I have enough info.

FINAL ANSWER: The weather in Lahore right now is sunny with a temperature of 34°C. 

Based on this sunny weather, here is a fun activity idea: **Great day for a walk in the park.**


**🔧 Checkpoint (3 min):** write a query that should trigger the `calculate` tool instead
(e.g. a hotel-cost-per-night question) and confirm the trace shows `calculate` being called.


In [27]:
# 🔧 write a query that should trigger calculate, then run it
answer = run_agent("If a hotel room costs 4500 PKR per night, how much for 5 nights?",
                    config=config, tool_registry=tool_registry)
print(answer)

[Step 1] Action: calculate({'expression': '4500 * 5'})
[Step 1] Observation: {'expression': '4500 * 5', 'result': 22500}
[Step 2] Thought: I have enough info.
A hotel room costing 4,500 PKR per night for 5 nights will be **22,500 PKR**.


---
## 🏆 Section 5 — Independent Challenge: Currency Converter (20 minutes)

No spec walkthrough this time — just the requirement. Build a 4th tool, `convert_currency`,
completely on your own, following the same pattern you've now done twice.

**Requirements:**
- Function: `convert_currency(amount: float, from_currency: str, to_currency: str) -> dict`
- Use this fake rate table (hardcode it, real-time rates are out of scope today):
  `{("PKR","USD"): 0.0036, ("USD","PKR"): 278.0, ("PKR","AED"): 0.013, ("AED","PKR"): 76.0}`
- Handle currency pairs not in the table gracefully (return an error dict, don't crash)
- Round the converted amount to 2 decimal places
- Write its `FunctionDeclaration`, register it alongside your other 3 tools, and test it with:
  *"How much is 8500 PKR in USD?"*


In [28]:
# ✍️ WRITE THIS: the whole convert_currency tool, end to end —
# function, FunctionDeclaration, registration, and a test call.
# (Everything you need is exactly what you did in Sections 2 and 4, just on your own this time.)
def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    rates = {
        ("PKR", "USD"): 0.0036,
        ("USD", "PKR"): 278.0,
        ("PKR", "AED"): 0.013,
        ("AED", "PKR"): 76.0
    }
    key = (from_currency.upper(), to_currency.upper())
    if key not in rates:
        return {"error": f"No conversion rate available for {from_currency} to {to_currency}"}
    converted = round(amount * rates[key], 2)
    return {"amount": amount, "from_currency": from_currency, "to_currency": to_currency, "converted_amount": converted}


convert_currency_declaration = types.FunctionDeclaration(
    name="convert_currency",
    description="Convert an amount from one currency to another using fixed exchange rates.",
    parameters={
        "type": "object",
        "properties": {
            "amount": {"type": "number", "description": "Amount to convert"},
            "from_currency": {"type": "string", "description": "Source currency code, e.g. 'PKR'"},
            "to_currency": {"type": "string", "description": "Target currency code, e.g. 'USD'"}
        },
        "required": ["amount", "from_currency", "to_currency"]
    }
)

multi_tool = types.Tool(function_declarations=[
    get_weather_declaration, calculate_declaration, suggest_activity_declaration, convert_currency_declaration
])
config = types.GenerateContentConfig(tools=[multi_tool])
tool_registry = {
    "get_weather": get_weather,
    "calculate": calculate,
    "suggest_activity": suggest_activity,
    "convert_currency": convert_currency
}

answer = run_agent("How much is 8500 PKR in USD?", config=config, tool_registry=tool_registry)
print(answer)

[Step 1] Action: convert_currency({'amount': 8500, 'to_currency': 'USD', 'from_currency': 'PKR'})
[Step 1] Observation: {'amount': 8500, 'from_currency': 'PKR', 'to_currency': 'USD', 'converted_amount': 30.6}
[Step 2] Thought: I have enough info.
8,500 PKR is equivalent to $30.60 USD (based on the current exchange rate).
